# 2606-Data Ecosystems and Governance in Organizations

# 0. Load Data

In [24]:
from pymongo import MongoClient
import pandas as pd
import json
from collections import Counter


# Load raw JSON into MongoDB
with open('data/raw_credit_applications.json') as f:
    data = json.load(f)

client = MongoClient()
db = client['novacred']
collection = db['applications']
collection.drop()  

# Find duplicates before inserting
ids = [record['_id'] for record in data]
id_counts = Counter(ids)
duplicate_ids = {id_: count for id_, count in id_counts.items() if count > 1}

print(f"Total records in file : {len(data)}")
print(f"Unique _id values     : {len(set(ids))}")
print(f"Duplicate IDs         : {list(duplicate_ids.keys())}")

# Show duplicates
for dup_id in duplicate_ids:
    versions = [r for r in data if r['_id'] == dup_id]
    print(f"\n--- Versions of {dup_id} ---")
    for i, v in enumerate(versions):
        print(f"  Version {i+1}: notes='{v.get('notes', 'none')}', keys={sorted(v.keys())}")


# Upsert into MongoDB
for record in data:
    collection.replace_one({'_id': record['_id']}, record, upsert=True)

print(f"\nDocuments in MongoDB after upsert: {collection.count_documents({})}")

Total records in file : 502
Unique _id values     : 500
Duplicate IDs         : ['app_042', 'app_001']

--- Versions of app_042 ---
  Version 1: notes='none', keys=['_id', 'applicant_info', 'decision', 'financials', 'spending_behavior']
  Version 2: notes='RESUBMISSION', keys=['_id', 'applicant_info', 'decision', 'financials', 'notes', 'spending_behavior']

--- Versions of app_001 ---
  Version 1: notes='none', keys=['_id', 'applicant_info', 'decision', 'financials', 'spending_behavior']
  Version 2: notes='DUPLICATE_ENTRY_ERROR', keys=['_id', 'applicant_info', 'decision', 'financials', 'notes', 'spending_behavior']

Documents in MongoDB after upsert: 500


# Check Duplicats

In [25]:
import pprint
pprint.pprint(collection.find_one({'_id': 'app_001'}))
print()
pprint.pprint(collection.find_one({'_id': 'app_042'}))

{'_id': 'app_001',
 'applicant_info': {'email': 'stephanie.nguyen47@mail.com',
                    'full_name': 'Stephanie Nguyen'},
 'decision': {'loan_approved': False, 'rejection_reason': 'high_dti_ratio'},
 'financials': {'annual_income': 102000,
                'credit_history_months': 37,
                'debt_to_income': 0.42,
                'savings_balance': 0},
 'notes': 'DUPLICATE_ENTRY_ERROR',
 'spending_behavior': [{'amount': 576, 'category': 'Fitness'}]}

{'_id': 'app_042',
 'applicant_info': {'date_of_birth': '1990-05-04',
                    'email': 'joseph.lopez1@gmail.com',
                    'full_name': 'Joseph Lopez',
                    'gender': 'Male',
                    'ip_address': '192.168.91.142',
                    'ssn': '652-70-5530',
                    'zip_code': '10044'},
 'decision': {'loan_approved': False,
              'rejection_reason': 'algorithm_risk_score'},
 'financials': {'annual_income': 69000,
                'credit_history_months'

**Notes**:

- The duplicate records were first identified when a BulkWriteError occurred during MongoDB ingestion. This demonstrates how database-level constraints can effectively serve as an early data quality gate. The raw dataset contained 502 records, of which 500 were unique applications. app_042 was flagged as a resubmission, and app_001 was flagged as a duplicate entry error. Each was resolved according to its nature: the resubmission retained the most recent version to honor the applicant's latest intent, and the duplicate entry error was resolved by restoring the original record.

In [ ]:
# Fix duplicte entry, keep correct version
# app_001 — restore original, discard DUPLICATE_ENTRY_ERROR version
original_app_001 = next(r for r in data 
                        if r['_id'] == 'app_001' 
                        and 'DUPLICATE_ENTRY_ERROR' not in str(r.get('notes', '')))

# Change to correct version
collection.replace_one({'_id': 'app_001'}, original_app_001, upsert=True)

print(f"\nFinal document count in MongoDB: {collection.count_documents({})}")


Final document count in MongoDB: 500
